In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import MNIST

In [4]:
# Importing the Dataset

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),   # also scales the images
    transforms.Normalize((0.5,), (0.5,)) # these are for RGB values, and since
                                         # this dataset is in greyscale, only one value is inside the bracket.
]
)

trainset = MNIST(root="./data", train = True, download = False, transform = transform)
testset = MNIST(root="./data", train = False, download = False, transform = transform)

In [5]:
# Creating DataLoader objects for training and testing datasets

trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64)

## CNN

In [6]:
# Defining the model

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()


        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

        )

        self.fc_layers = nn.Sequential(
            nn.Linear(3*3*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)

        return x

In [7]:
# Creating a model object

CNN_model = CNN()

In [8]:
# Defining the Loss function and Optimizer function.

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(CNN_model.parameters())

In [10]:
# training
epochs = 10

for epoch in range(epochs):
    CNN_model.train()
    
    epoch_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = CNN_model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    loss = epoch_loss/len(trainloader)

    print(f'for epoch {epoch+1}/{epochs}, loss = {loss}')

for epoch 1/10, loss = 0.05233165149146052
for epoch 2/10, loss = 0.033293462152547415
for epoch 3/10, loss = 0.026062863667018717
for epoch 4/10, loss = 0.018885258329788168
for epoch 5/10, loss = 0.016807873453069944
for epoch 6/10, loss = 0.013497006896781781
for epoch 7/10, loss = 0.011520504360959012
for epoch 8/10, loss = 0.009803108727635415
for epoch 9/10, loss = 0.00914334201168236
for epoch 10/10, loss = 0.00870615407254713


In [11]:
# Testing the model
CNN_model.eval()

total_values = 0
correct_values = 0

with torch.no_grad():
    
    for images, labels in testloader:
        output = CNN_model(images)
        _, predicted = torch.max(output, 1)
        
        total_values += labels.size(0)
        correct_values += (predicted == labels).sum().item()

print(f'Model Accuracy = {correct_values/total_values * 100} ')

Model Accuracy = 99.25 


## RNN

In [12]:
# Defining the Model 
class RNN(nn.Module):
    def __init__(self, input_size = 28, hidden_size = 128, num_layers = 2, num_classes = 10):
        super().__init__()

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first = True)

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = x.squeeze(dim = 1)

        out, _ = self.rnn(x)

        last_time_step = out[:, -1, :]

        return self.fc(last_time_step)

In [13]:
# Creating a model object

RNN_model = RNN()
# Defining the Loss function and Optimizer function.

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(RNN_model.parameters(), lr = 0.001)

In [14]:
# Training the model

epochs = 10

for epoch in range(epochs):
    RNN_model.train()
    epoch_loss = 0.0
    
    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = RNN_model(images)
        
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()

    final_loss = epoch_loss/len(trainloader)
    print(f'for epoch = {epoch + 1}/{epochs}, loss = {final_loss}')

for epoch = 1/10, loss = 0.6054803332301981
for epoch = 2/10, loss = 0.26780202005630427
for epoch = 3/10, loss = 0.18491148527489223
for epoch = 4/10, loss = 0.16615707161369672
for epoch = 5/10, loss = 0.14824204518001027
for epoch = 6/10, loss = 0.14024584029596657
for epoch = 7/10, loss = 0.12758083747568796
for epoch = 8/10, loss = 0.13411317043590235
for epoch = 9/10, loss = 0.11732548300816274
for epoch = 10/10, loss = 0.10851072151162254


In [15]:
# Evaluating / Testing the model
RNN_model.eval()

total_vals = 0
correct_vals = 0
with torch.no_grad():
    for images, labels in testloader:
        output = RNN_model(images)

        _, predicted = output.max(1)
        
        total_vals += labels.size(0)
        correct_vals += (predicted.eq(labels)).sum().item()

print(f'Model Accuracy = {correct_vals/total_vals *100}%')

Model Accuracy = 97.0%
